# 2. Treinamento e avaliação
Treina Random Forest e XGBoost, escolhe o threshold do XGBoost apenas com o treino, avalia o teste e salva os artefatos. Métricas incluem relatório por classe, accuracy, matriz de confusão, AUC-ROC e validação cruzada.

In [1]:
import json
import joblib
from sklearn.model_selection import cross_val_score
from breast_cancer.config import (DATA_PATH, MODEL_DIR, REPORT_DIR, FIGURE_DIR,
                                  RANDOM_STATE, THRESHOLD_MIN_PRECISION)
from breast_cancer.data import load_data
from breast_cancer.preprocessing import prepare_data
from breast_cancer.models import make_xgboost, make_random_forest
from breast_cancer.evaluation import select_threshold, evaluate_model
from breast_cancer.visualization import save_roc_curve

X_train, X_test, y_train, y_test = prepare_data(load_data(DATA_PATH))
print(f'Treino: {X_train.shape}; teste: {X_test.shape}')

Treino: (341, 30); teste: (228, 30)


In [2]:
xgboost = make_xgboost()
cv_scores = cross_val_score(xgboost, X_train, y_train, cv=3, scoring='accuracy', n_jobs=1)
xgboost.fit(X_train, y_train)
train_probabilities = xgboost.predict_proba(X_train)[:, 1]
threshold = select_threshold(y_train, train_probabilities, THRESHOLD_MIN_PRECISION)
test_probabilities = xgboost.predict_proba(X_test)[:, 1]
xgb_metrics = evaluate_model(y_test, test_probabilities, threshold)
print('XGBoost | CV (3 folds):', cv_scores)
print(f"Threshold: {threshold:.3f}; AUC-ROC: {xgb_metrics['auc_roc']:.3f}; Accuracy: {xgb_metrics['accuracy']:.3f}")
print(xgb_metrics['classification_report'])
print('Matriz de confusão (linhas reais, colunas previstas):\n', xgb_metrics['confusion_matrix'])

XGBoost | CV (3 folds): [0.95614035 0.94736842 0.94690265]
Threshold: 0.139; AUC-ROC: 0.997; Accuracy: 0.965
              precision    recall  f1-score   support

     Benigno      0.993     0.951     0.971       143
     Maligno      0.923     0.988     0.955        85

    accuracy                          0.965       228
   macro avg      0.958     0.970     0.963       228
weighted avg      0.967     0.965     0.965       228

Matriz de confusão (linhas reais, colunas previstas):
 [[136   7]
 [  1  84]]


In [3]:
forest = make_random_forest()
forest_cv = cross_val_score(forest, X_train, y_train, cv=3, scoring='accuracy', n_jobs=1)
forest.fit(X_train, y_train)
forest_probabilities = forest.predict_proba(X_test)[:, 1]
forest_metrics = evaluate_model(y_test, forest_probabilities, threshold=0.5)
print('Random Forest | CV (3 folds):', forest_cv)
print(f"AUC-ROC: {forest_metrics['auc_roc']:.3f}; Accuracy: {forest_metrics['accuracy']:.3f}")
print(forest_metrics['classification_report'])
print('Matriz de confusão:\n', forest_metrics['confusion_matrix'])

Random Forest | CV (3 folds): [0.95614035 0.94736842 0.92035398]
AUC-ROC: 0.994; Accuracy: 0.969
              precision    recall  f1-score   support

     Benigno      0.959     0.993     0.976       143
     Maligno      0.988     0.929     0.958        85

    accuracy                          0.969       228
   macro avg      0.973     0.961     0.967       228
weighted avg      0.970     0.969     0.969       228

Matriz de confusão:
 [[142   1]
 [  6  79]]


In [4]:
save_roc_curve(xgb_metrics['fpr'], xgb_metrics['tpr'], xgb_metrics['auc_roc'], FIGURE_DIR / 'curva_roc.png')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump({'model': xgboost, 'threshold': threshold, 'feature_names': list(X_train.columns)}, MODEL_DIR / 'xgboost.joblib')
joblib.dump(forest, MODEL_DIR / 'random_forest.joblib')
summary = {
    'xgboost': {'threshold': threshold, 'auc_roc': xgb_metrics['auc_roc'], 'accuracy': xgb_metrics['accuracy'], 'cv_accuracy': cv_scores.tolist(), 'confusion_matrix': xgb_metrics['confusion_matrix'].tolist(), 'classification_report': xgb_metrics['classification_report']},
    'random_forest': {'auc_roc': forest_metrics['auc_roc'], 'accuracy': forest_metrics['accuracy'], 'cv_accuracy': forest_cv.tolist(), 'confusion_matrix': forest_metrics['confusion_matrix'].tolist(), 'classification_report': forest_metrics['classification_report']}
}
(REPORT_DIR / 'metrics.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Resultados salvos em {REPORT_DIR}')

Resultados salvos em C:\Users\mateu_g1p9kd2\OneDrive\Documentos\GitHub\Predicao-cancer-de-mama-vs2\reports


## Inferência com o modelo salvo
O artefato inclui modelo, threshold e nomes das features esperadas. Alinhe as colunas da entrada com `feature_names` antes de chamar `predict_proba`.

In [5]:
artifact = joblib.load(MODEL_DIR / 'xgboost.joblib')
artifact['model'].predict_proba(X_test.loc[:, artifact['feature_names']])[:3, 1]

array([0.99400926, 0.79233664, 0.01121518], dtype=float32)